# Project0 Colab Launcher

This notebook prepares a Google Colab GPU runtime, installs Project0 and Ollama, loads `qwen2.5:7b`, and opens the Project0 dashboard. Before running, select a GPU runtime.

## Step 1 - Clone or Update Project0

This step retrieves the Project0 source code from the public GitHub repository.

- If `/content/project0` does not exist, the public repository is cloned.
- If it already exists, the local repository is updated with `git pull --ff-only`.
- No GitHub account, access token, or Colab secret is required.

Afterward, the notebook changes the working directory to `/content/project0`.

In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/project0")
REPO_URL = "https://github.com/pgailinas/project0.git"

if not REPO_DIR.exists():
    print("Cloning Project0...")
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print("Project0 already exists. Updating repository...")
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )

%cd /content/project0

print("Project0 repository ready.")



## Step 2 - Install Project0

This step installs Project0 and its required Python dependencies into the current Colab runtime.

Project0 is installed in editable mode from `/content/project0`. The source files remain in the cloned repository and are not overwritten by this installation step.

In [ ]:
import sys
import subprocess

print("Python:", sys.version)

if sys.version_info < (3, 12):
    raise RuntimeError(
        "Project0 requires Python >= 3.12. "
        f"This Colab runtime is Python {sys.version_info.major}.{sys.version_info.minor}."
    )

print("\nInstalling Project0 from the cloned repository...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        "/content/project0",
    ],
    check=True,
)

print("\nProject0 installation complete.")

## Step 3 - Verify Colab GPU

This step confirms that the Colab runtime has a CUDA-capable GPU available and reports the GPU name and VRAM.

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is available. "
        "In Colab, select Runtime > Change runtime type > GPU."
    )

gpu_index = 0
gpu_name = torch.cuda.get_device_name(gpu_index)
gpu_props = torch.cuda.get_device_properties(gpu_index)
total_vram_gb = gpu_props.total_memory / (1024 ** 3)

print("GPU:", gpu_name)
print(f"VRAM: {total_vram_gb:.1f} GB")

## Step 4 - Install and Start Ollama

This step installs the `zstd` dependency and Ollama, starts `ollama serve` in the background, and confirms that the local API is responding at `127.0.0.1:11434`.

In [ ]:
!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time
import urllib.request

OLLAMA_LOG = "/content/ollama.log"

ollama_server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open(OLLAMA_LOG, "w"),
    stderr=subprocess.STDOUT,
)

print(f"Ollama server started (PID {ollama_server.pid}).")

ollama_ready = False
for attempt in range(15):
    time.sleep(1)
    try:
        response = urllib.request.urlopen(
            "http://127.0.0.1:11434/api/tags",
            timeout=5,
        )
        print("Ollama server: RUNNING")
        print("HTTP status:", response.status)
        ollama_ready = True
        break
    except Exception:
        pass

if not ollama_ready:
    raise RuntimeError(
        "Ollama server did not start.\n\n" + open(OLLAMA_LOG).read()
    )

## Step 5 - Load and Verify `qwen2.5:7b`

This step pulls the Project0 model, lists the installed Ollama models, runs a short test inference, and displays the active Ollama processor assignment.

After the test inference, the `PROCESSOR` column reported by `ollama ps` should indicate GPU use.

In [ ]:
!ollama pull qwen2.5:7b
!ollama list
!ollama run qwen2.5:7b "Reply with exactly: Ollama GPU test successful."
!ollama ps

## Step 6 - Configure and Start Project0

This step configures the Project0 runtime and starts the dashboard server as a background process inside the Colab runtime.

The launcher explicitly configures:

- Ollama as the reasoning provider
- `qwen2.5:7b` as the Documentation Agent reasoning model
- OpenAlex, Crossref, arXiv, and OpenReview as Research Agent sources
- DEBUG logging for development and diagnostics

Project0 listens internally at `http://127.0.0.1:8001`.

In [ ]:
import os
import subprocess
import time
import urllib.request
from pathlib import Path
from datetime import datetime

# ============================================================
# Project0 Colab Debug Configuration
# ============================================================

PROJECT0_ROOT = "/content/project0"
PROJECT0_LOG = "/content/project0_dashboard.log"

# Optional artifact directory for this Colab session
DEBUG_ARTIFACT_ROOT = Path("/content/project0_debug_artifacts")
DEBUG_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

# Unique identifier for this test iteration
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Project0 debug iteration: {RUN_ID}")


# ============================================================
# Stop any previous Project0 dashboard process
# ============================================================

subprocess.run(
    [
        "pkill",
        "-f",
        "project0.dashboard.dashboard_app",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(1)


# ============================================================
# Clear the dashboard log
#
# This preserves the same workflow used locally:
# every test iteration begins with a clean terminal log.
# ============================================================

Path(PROJECT0_LOG).write_text("")


# ============================================================
# Project0 runtime configuration
# ============================================================

os.environ["PROJECT0_REASONING_PROVIDER"] = "ollama"
os.environ["PROJECT0_DOCUMENTATION_OLLAMA_MODEL"] = "qwen2.5:7b"

os.environ["PROJECT0_RESEARCH_SOURCE_PROVIDERS"] = (
    "openalex,crossref,arxiv,openreview"
)

os.environ["PROJECT0_LOG_LEVEL"] = "DEBUG"


print("\nProject0 configuration:")
print(
    "  Reasoning provider:",
    os.environ["PROJECT0_REASONING_PROVIDER"],
)
print(
    "  Ollama model:",
    os.environ["PROJECT0_DOCUMENTATION_OLLAMA_MODEL"],
)
print(
    "  Research sources:",
    os.environ["PROJECT0_RESEARCH_SOURCE_PROVIDERS"],
)
print(
    "  Log level:",
    os.environ["PROJECT0_LOG_LEVEL"],
)


# ============================================================
# Start Project0 dashboard
# ============================================================

log_file = open(
    PROJECT0_LOG,
    "w",
    buffering=1,
)

dashboard = subprocess.Popen(
    [
        "python",
        "-m",
        "project0.dashboard.dashboard_app",
    ],
    cwd=PROJECT0_ROOT,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

print(
    f"\nProject0 process started "
    f"(PID {dashboard.pid})."
)

print("Waiting for dashboard...")


# ============================================================
# Wait for dashboard
# ============================================================

dashboard_ready = False

for attempt in range(15):
    time.sleep(1)

    try:
        response = urllib.request.urlopen(
            "http://127.0.0.1:8001",
            timeout=2,
        )

        print("Project0 Dashboard: RUNNING")
        print("HTTP status:", response.status)

        dashboard_ready = True
        break

    except Exception:
        pass


if not dashboard_ready:
    print("Project0 dashboard did not start.")

    print("\n===== Project0 log =====")

    log_text = Path(PROJECT0_LOG).read_text(
        errors="replace"
    )

    print(log_text)



## Step 7 - Open the Project0 Dashboard

Project0 runs inside the Colab runtime on `127.0.0.1:8001`.

This step installs Cloudflare's `cloudflared` utility and creates a temporary
HTTPS tunnel to the Project0 Dashboard. No Cloudflare account or credentials
are required.

The generated `trycloudflare.com` URL remains valid only while this Colab
runtime and the tunnel process are running.

Open the displayed **Project0 Dashboard** link in a new browser tab.

In [ ]:
from pathlib import Path
import re
import subprocess
import time
from IPython.display import display, HTML

CLOUDFLARED_PATH = Path("/usr/local/bin/cloudflared")

# ---------------------------------------------------------------------
# Install cloudflared if necessary.
# ---------------------------------------------------------------------

if not CLOUDFLARED_PATH.exists():
    print("Installing cloudflared...")

    subprocess.run(
        [
            "wget",
            "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O",
            str(CLOUDFLARED_PATH),
        ],
        check=True,
    )

    subprocess.run(
        ["chmod", "+x", str(CLOUDFLARED_PATH)],
        check=True,
    )

print(
    subprocess.run(
        [str(CLOUDFLARED_PATH), "--version"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
)

# ---------------------------------------------------------------------
# Stop a tunnel previously started by this notebook cell, if present.
# This makes the cell safe to rerun.
# ---------------------------------------------------------------------

if "cloudflared_process" in globals():
    if cloudflared_process.poll() is None:
        print("Stopping existing Project0 tunnel...")
        cloudflared_process.terminate()

        try:
            cloudflared_process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            cloudflared_process.kill()
            cloudflared_process.wait()

# ---------------------------------------------------------------------
# Start a Cloudflare Quick Tunnel to Project0.
# ---------------------------------------------------------------------

print("Starting Project0 Cloudflare tunnel...")

cloudflared_process = subprocess.Popen(
    [
        str(CLOUDFLARED_PATH),
        "tunnel",
        "--url",
        "http://127.0.0.1:8001",
        "--no-autoupdate",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

tunnel_url = None
deadline = time.time() + 45

while time.time() < deadline:
    line = cloudflared_process.stdout.readline()

    if line:
        print(line.rstrip())

        match = re.search(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            line,
        )

        if match:
            tunnel_url = match.group(0)
            break

    if cloudflared_process.poll() is not None:
        break

if tunnel_url is None:
    if cloudflared_process.poll() is None:
        cloudflared_process.terminate()

    raise RuntimeError(
        "Cloudflare tunnel did not provide a dashboard URL. "
        "Verify that Project0 is running on 127.0.0.1:8001 "
        "and rerun this cell."
    )

# ---------------------------------------------------------------------
# Display the Dashboard link.
# ---------------------------------------------------------------------

print()
print("Project0 Dashboard ready:")
print(tunnel_url)

display(
    HTML(
        f"""
        <p>
          <a href="{tunnel_url}"
             target="_blank"
             rel="noopener noreferrer"
             style="font-size:18px;font-weight:bold;">
             Open Project0 Dashboard
          </a>
        </p>
        """
    )
)



## Step 8 - Stop Project0 (Optional)

Use this optional cleanup step when you are finished with the Project0 Dashboard.

By default, cleanup is skipped so that **Run All** leaves the Dashboard and its public link running.

To stop Project0, enable the `STOP_PROJECT0` checkbox and run this cell manually. The cleanup stops:

- The Cloudflare Quick Tunnel created in Step 7
- The Project0 Dashboard process created in Step 6

To stop the entire Colab session and remove all files under `/content`, use **Runtime → Disconnect and delete runtime**.

In [ ]:
import subprocess

STOP_PROJECT0 = False  # @param {type:"boolean"}

if not STOP_PROJECT0:
    print("Project0 cleanup skipped.")
else:
    # Stop the Cloudflare Quick Tunnel.
    if (
        "cloudflared_process" in globals()
        and cloudflared_process.poll() is None
    ):
        print("Stopping Project0 Cloudflare tunnel...")
        cloudflared_process.terminate()

        try:
            cloudflared_process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            print(
                "Cloudflare tunnel did not stop normally; "
                "terminating it..."
            )
            cloudflared_process.kill()
            cloudflared_process.wait()

        print("Cloudflare tunnel stopped.")
    else:
        print("Cloudflare tunnel is not running.")

    # Stop the Project0 Dashboard.
    if "dashboard" in globals() and dashboard.poll() is None:
        print("Stopping Project0 Dashboard...")
        dashboard.terminate()

        try:
            dashboard.wait(timeout=10)
        except subprocess.TimeoutExpired:
            print(
                "Project0 did not stop normally; terminating it..."
            )
            dashboard.kill()
            dashboard.wait()

        print("Project0 Dashboard stopped.")
    else:
        print("Project0 Dashboard is not running.")

    print()
    print("Project0 Colab session cleanup complete.")

